# Orbit Wars Tutorial

Conquer planets rotating around a sun! Players send fleets of ships between planets to capture territory in a continuous 100x100 space.

## Game Mechanics

- **Planets** produce ships each turn (proportional to their radius)
- **Inner planets** rotate around the central sun; outer planets are static
- **Fleets** fly in straight lines at a given angle from their source planet
- **Fleet speed** scales with fleet size (1 ship = 1/turn, larger fleets up to 6/turn)
- **Combat**: arriving fleet ships are subtracted from the planet's garrison. If the garrison drops below 0, ownership flips
- **Sun**: fleets that hit the sun are destroyed
- **Comets**: temporary planets that fly through the board on elliptical paths
- **Win condition**: most ships (planets + fleets) when time runs out or only one player remains

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

## Understanding the Observation

Each turn your agent receives an observation with:
- `player` - your player ID (0-3)
- `planets` - list of `[id, owner, x, y, radius, ships, production]`
- `fleets` - list of `[id, owner, x, y, angle, from_planet_id, ships]`
- `angular_velocity` - how fast inner planets rotate (radians/turn)

Your agent returns a list of moves: `[from_planet_id, angle_in_radians, num_ships]`

In [ ]:
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

## Agent 1: Nearest Planet Sniper

Our first agent uses a simple strategy:
1. For each planet we own, find the nearest planet we **don't** own
2. Check if we have enough ships to capture it (need more than the target's garrison)
3. Send exactly enough ships to take it (target ships + 1)

This teaches the fundamentals: reading observations, computing angles, and sending fleets.

In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_sniper(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    # Separate our planets from targets
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        # Find the nearest planet we don't own
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        # How many ships do we need? Target's garrison + 1
        ships_needed = max(nearest.ships + 1, 20)

        # Only send if we have enough
        if mine.ships >= ships_needed:
            # Calculate angle from our planet to the target
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [ ]:
# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([nearest_planet_sniper, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="ipython", width=800, height=600)

## What's wrong with this agent?

The sniper agent has a few problems:
- It doesn't account for **travel time** -- the target planet produces ships while the fleet is in transit
- It sends fleets from **every** planet, even if multiple are targeting the same planet
- It ignores the **sun** -- fleets aimed through the center get destroyed
- It holds ships on planets that have no nearby targets instead of consolidating

Let's try it in a 4-player game to see how it holds up:

In [ ]:
env4 = make("orbit_wars", debug=True)
env4.run([nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper, nearest_planet_sniper])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env4.render(mode="ipython", width=800, height=600)

## Making a submission

You can either submit a main.py, a tar.gz (or zip) with a main.py in it, or submit a notebook with a main.py or submission.tar.gz

There are three ways to subit.
1. using the [Submit Agent](https://www.kaggle.com/competitions/orbit-wars) button on the homepage and uploading the file
2. using the Kaggle CLI (as described in agents.py in the competition dataset)
3. submitting a notebook with a main.py or submission.tar.gz

In [ ]:
import math
from collections import defaultdict

# ============================================================
# DATA CLASSES
# ============================================================
class PlanetObj:
    def __init__(self, id, owner, x, y, radius, ships, production):
        self.id = id
        self.owner = owner
        self.x = float(x)
        self.y = float(y)
        self.radius = float(radius)
        self.ships = float(ships)
        self.production = float(production)


class FleetObj:
    def __init__(self, id, owner, x, y, angle, from_planet_id, ships):
        self.id = id
        self.owner = owner
        self.x = float(x)
        self.y = float(y)
        self.angle = float(angle)
        self.from_planet_id = from_planet_id
        self.ships = float(ships)


# ============================================================
# CONSTANTS
# ============================================================
BOARD = 100.0
SUN_X, SUN_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
TOTAL_STEPS = 500
HORIZON = 85
ATTACK_HORIZON = 70
DEF_HORIZON = 34
ORBIT_LIMIT = 48.0
SUN_SAFETY = 1.55
PLANET_MARGIN = 0.35

# Strategic knobs
EARLY_STEP = 42
MID_STEP = 150
MIN_LAUNCH = 5
MIN_RESERVE = 7
MAX_LAUNCHES_PER_SRC = 3
GLOBAL_ATTACK_RATIO_EARLY = 0.34
GLOBAL_ATTACK_RATIO_MID = 0.45
GLOBAL_ATTACK_RATIO_LATE = 0.60


# ============================================================
# SMALL UTILS
# ============================================================
def getv(obs, key, default=None):
    if isinstance(obs, dict):
        return obs.get(key, default)
    return getattr(obs, key, default)


def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


# ============================================================
# PHYSICS
# ============================================================
def fleet_speed(ships):
    ships = max(1, int(ships))
    if ships <= 1:
        return 1.0
    ratio = math.log(ships) / math.log(1000.0)
    ratio = clamp(ratio, 0.0, 1.0)
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)


def travel_time(x1, y1, r1, x2, y2, r2, ships):
    d = max(math.hypot(x2 - x1, y2 - y1) - r1 - r2, 0.0)
    return d / fleet_speed(max(1, ships))


def seg_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    ls = dx * dx + dy * dy
    if ls <= 1e-12:
        return math.hypot(px - x1, py - y1)
    t = ((px - x1) * dx + (py - y1) * dy) / ls
    t = clamp(t, 0.0, 1.0)
    cx = x1 + t * dx
    cy = y1 + t * dy
    return math.hypot(px - cx, py - cy)


def sun_blocked(x1, y1, x2, y2, margin=SUN_SAFETY):
    return seg_dist(SUN_X, SUN_Y, x1, y1, x2, y2) < SUN_R + margin


def is_orbiting(p):
    return math.hypot(p.x - SUN_X, p.y - SUN_Y) + p.radius < ORBIT_LIMIT


def predict_orbit(px, py, omega, dt):
    r = math.hypot(px - SUN_X, py - SUN_Y)
    th = math.atan2(py - SUN_Y, px - SUN_X)
    return SUN_X + r * math.cos(th + omega * dt), SUN_Y + r * math.sin(th + omega * dt)


def normalize_angle(a):
    while a <= -math.pi:
        a += 2 * math.pi
    while a > math.pi:
        a -= 2 * math.pi
    return a


def safe_angle(x1, y1, x2, y2):
    direct = math.atan2(y2 - y1, x2 - x1)
    if not sun_blocked(x1, y1, x2, y2):
        return direct

    d = math.hypot(x1 - SUN_X, y1 - SUN_Y)
    if d <= SUN_R + 1.0:
        return direct

    half = math.asin(min(1.0, (SUN_R + 2.0) / d))
    to_sun = math.atan2(SUN_Y - y1, SUN_X - x1)
    cw = to_sun + half
    ccw = to_sun - half

    def adiff(a):
        dd = normalize_angle(a - direct)
        return abs(dd)

    return cw if adiff(cw) < adiff(ccw) else ccw


# ============================================================
# INTERCEPT SOLVER
# ============================================================
def solve_intercept(src, tgt, omega, ships, orbiting):
    """Return predicted intercept point + ETA.
    Stable fixed-point iteration for orbiting planets.
    """
    ships = max(1, int(ships))

    if not orbiting:
        t = travel_time(src.x, src.y, src.radius, tgt.x, tgt.y, tgt.radius, ships)
        return tgt.x, tgt.y, t

    t = travel_time(src.x, src.y, src.radius, tgt.x, tgt.y, tgt.radius, ships)
    ix, iy = tgt.x, tgt.y

    # Fixed-point refinement. Keep it small and stable.
    for _ in range(12):
        ix, iy = predict_orbit(tgt.x, tgt.y, omega, t)
        nt = travel_time(src.x, src.y, src.radius, ix, iy, tgt.radius, ships)
        if abs(nt - t) < 0.03:
            t = nt
            break
        t = 0.6 * t + 0.4 * nt

    # If the predicted intercept is effectively impossible because the path crosses the sun,
    # the caller will often reject it. We keep the solver clean and deterministic.
    return ix, iy, t


# ============================================================
# PATH / TARGETING
# ============================================================
def planet_blocked(x1, y1, x2, y2, planets, src_id, tgt_id):
    """Coarse obstacle filter: ignore orbiting planets as hard blockers to avoid over-pruning."""
    for p in planets:
        if p.id == src_id or p.id == tgt_id:
            continue
        if is_orbiting(p):
            continue
        if seg_dist(p.x, p.y, x1, y1, x2, y2) < p.radius + PLANET_MARGIN:
            return True
    return False


def fleet_target_planet(fleet, planets):
    """Estimate which planet a fleet is heading toward based on its ray direction.
    This is used for arrival ledger construction.
    """
    best_planet, best_time = None, 1e18
    fx, fy = math.cos(fleet.angle), math.sin(fleet.angle)
    sp = fleet_speed(fleet.ships)

    for p in planets:
        dx = p.x - fleet.x
        dy = p.y - fleet.y
        proj = dx * fx + dy * fy
        if proj < 0:
            continue

        perp_sq = dx * dx + dy * dy - proj * proj
        rad_sq = p.radius * p.radius
        if perp_sq >= rad_sq:
            continue

        hit_d = max(0.0, proj - math.sqrt(max(0.0, rad_sq - perp_sq)))
        turns = hit_d / sp
        if turns <= HORIZON and turns < best_time:
            best_time = turns
            best_planet = p

    if best_planet is None:
        return None, None
    return best_planet, int(math.ceil(best_time))


# ============================================================
# COMBAT SIMULATION
# ============================================================
def resolve_combat(owner, garrison, arrivals_at_turn):
    """Same-turn arrivals: friendly reinforce, then attackers resolve.
    Kept compatible with your original logic.
    """
    by_owner = {}
    for _, o, s in arrivals_at_turn:
        by_owner[o] = by_owner.get(o, 0) + int(s)

    # Friendly ships reinforce first
    if owner in by_owner:
        garrison += by_owner.pop(owner)

    if not by_owner:
        return owner, garrison

    attackers = sorted(by_owner.items(), key=lambda x: -x[1])
    top_o, top_s = attackers[0]
    second_s = attackers[1][1] if len(attackers) > 1 else 0
    effective = top_s - second_s

    if effective <= 0:
        return owner, garrison

    if effective > garrison:
        return top_o, effective - garrison
    return owner, garrison - effective


def simulate_timeline(planet, all_arrivals, horizon):
    horizon = max(1, int(horizon))
    events = defaultdict(list)
    for eta, o, s in all_arrivals:
        t = max(1, int(math.ceil(eta)))
        if t <= horizon and s > 0:
            events[t].append((t, o, int(s)))

    owner = planet.owner
    garrison = float(planet.ships)
    owner_at = {0: owner}
    ships_at = {0: garrison}

    for turn in range(1, horizon + 1):
        if owner != -1:
            garrison += planet.production
        if turn in events:
            owner, garrison = resolve_combat(owner, garrison, events[turn])
            garrison = max(0.0, garrison)
        owner_at[turn] = owner
        ships_at[turn] = garrison

    return owner_at, ships_at


def safe_ships_available(planet, arrivals, player, horizon):
    """Minimum garrison that remains at any point in the future if we do nothing."""
    if planet.owner != player:
        return 0

    garrison = float(planet.ships)
    min_future = garrison
    events = defaultdict(list)
    for eta, o, s in arrivals:
        t = max(1, int(math.ceil(eta)))
        if t <= horizon and s > 0:
            events[t].append((t, o, int(s)))

    for turn in range(1, horizon + 1):
        if planet.owner != -1:
            garrison += planet.production
        if turn in events:
            garrison_owner = planet.owner
            garrison, garrison_owner = garrison, garrison_owner
            garrison_owner, garrison = resolve_combat(garrison_owner, garrison, events[turn])
            garrison = max(0.0, garrison)
        min_future = min(min_future, garrison)

    return max(0.0, min_future)


def ships_needed_at_eta(planet, eta, player, base_arrivals, planned_extra):
    """Minimal ships needed at turn eta to end that turn owning the planet.
    Uses exact simulation + binary search.
    """
    eta = max(1, int(math.ceil(eta)))

    all_base = [
        (int(math.ceil(t)), o, int(s))
        for t, o, s in (base_arrivals + list(planned_extra))
        if int(math.ceil(t)) <= eta and s > 0
    ]

    oa, _ = simulate_timeline(planet, all_base, eta)
    if oa.get(eta, planet.owner) == player:
        return 0

    def owns_with(extra_ships):
        test = all_base + [(eta, player, int(extra_ships))]
        oa2, _ = simulate_timeline(planet, test, eta)
        return oa2.get(eta, planet.owner) == player

    hi = max(1, int(planet.ships) + int(planet.production * eta) + 4)
    cap = 5000
    while hi <= cap and not owns_with(hi):
        hi *= 2
    if hi > cap:
        return cap + 1

    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if owns_with(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


# ============================================================
# STRATEGIC HELPERS
# ============================================================
def phase_of_game(step, my_planets, enemy_planets, my_prod, enemy_prod):
    gp = step / float(TOTAL_STEPS)
    my_count = len(my_planets)
    enemy_count = len(enemy_planets)

    if step < EARLY_STEP or my_count <= 2:
        return "early"

    if gp > 0.82 or my_count >= 5 and my_prod >= enemy_prod * 1.15:
        return "late"

    if enemy_count == 0:
        return "cleanup"

    if my_prod < enemy_prod * 0.80 or my_count < enemy_count:
        return "defend"

    return "mid"


def relative_reaction_time(tgt, my_planets, enemy_planets):
    my_t = min((travel_time(p.x, p.y, p.radius, tgt.x, tgt.y, tgt.radius, max(1, p.ships)) for p in my_planets), default=1e18)
    en_t = min((travel_time(p.x, p.y, p.radius, tgt.x, tgt.y, tgt.radius, max(1, p.ships)) for p in enemy_planets), default=1e18)
    return my_t, en_t


def indirect_wealth(planet, all_planets, player):
    w = 0.0
    for p in all_planets:
        if p.id == planet.id:
            continue
        d = dist(planet.x, planet.y, p.x, p.y)
        if d < 1.0:
            continue
        factor = p.production / (d + 10.0)
        if p.owner == player:
            w += 0.45 * factor
        elif p.owner == -1:
            w += 1.00 * factor
        else:
            w += 1.30 * factor
    return w


# ============================================================
# MAIN AGENT
# ============================================================
def agent(obs):
    step = int(getv(obs, "step", 0) or 0)
    player = int(getv(obs, "player", 0) or 0)
    omega = float(getv(obs, "angular_velocity", 0.03) or 0.03)
    raw_planets = getv(obs, "planets", []) or []
    raw_fleets = getv(obs, "fleets", []) or []
    comet_ids = set(getv(obs, "comet_planet_ids", []) or [])

    planets = [PlanetObj(*p[:7]) for p in raw_planets]
    fleets = [FleetObj(*f[:7]) for f in raw_fleets]

    if not planets:
        return []

    my_planets = [p for p in planets if p.owner == player]
    enemy_planets = [p for p in planets if p.owner not in (player, -1)]
    neutral_planets = [p for p in planets if p.owner == -1]

    if not my_planets:
        return []

    my_prod = sum(p.production for p in my_planets)
    enemy_prod = sum(p.production for p in enemy_planets)
    phase = phase_of_game(step, my_planets, enemy_planets, my_prod, enemy_prod)
    gp = step / float(TOTAL_STEPS)
    remaining_steps = max(1, TOTAL_STEPS - step)

    # ---------- build arrival ledger ----------
    ledger = {p.id: [] for p in planets}
    for f in fleets:
        tgt, eta = fleet_target_planet(f, planets)
        if tgt is None:
            continue
        ledger[tgt.id].append((eta, f.owner, int(f.ships)))

    # ---------- defense / reserve calculations ----------
    planned_commits = defaultdict(list)   # target_id -> [(eta, owner, ships)]
    spent = {p.id: 0 for p in my_planets}
    moves = []

    # A planet-specific reserve: more conservative in early/mid, slightly looser late.
    def strategic_reserve(p):
        if phase == "early":
            return int(8 + 2.0 * p.production)
        if phase == "mid":
            return int(7 + 1.4 * p.production)
        if phase == "defend":
            return int(8 + 1.8 * p.production)
        if phase == "late":
            return int(5 + 1.0 * p.production)
        return int(6 + 1.2 * p.production)

    safe_available = {}
    base_reserve = {}
    for p in my_planets:
        safe_available[p.id] = safe_ships_available(p, ledger[p.id], player, DEF_HORIZON)
        base_reserve[p.id] = strategic_reserve(p)

    # ---------- emergency defense first ----------
    # Detect planets that are about to flip and reinforce them from the closest safe ally.
    threatened = []
    for mine in my_planets:
        all_arr = ledger[mine.id] + planned_commits[mine.id]
        oa, _ = simulate_timeline(mine, all_arr, min(28, ATTACK_HORIZON))
        fall_turn = None
        for t in range(1, min(28, ATTACK_HORIZON) + 1):
            if oa.get(t, player) != player:
                fall_turn = t
                break
        if fall_turn is not None:
            threatened.append((fall_turn, mine))

    threatened.sort(key=lambda x: x[0])

    for fall_turn, mine in threatened:
        # Already safe? skip.
        all_arr = ledger[mine.id] + planned_commits[mine.id]
        need = ships_needed_at_eta(
            mine,
            fall_turn,
            player,
            base_arrivals=ledger[mine.id],
            planned_extra=planned_commits[mine.id],
        )
        if need <= 0:
            continue

        # Need a little extra because arrival order and nearby fights can be noisy.
        need = int(need + mine.production + 2)

        donors = sorted(
            [p for p in my_planets if p.id != mine.id],
            key=lambda a: dist(a.x, a.y, mine.x, mine.y),
        )

        for ally in donors:
            if sum(s for _, o, s in ledger[ally.id] if o != player) > 0 and fall_turn <= 10:
                # Do not weaken a planet that is already under heavy attack in a near-term emergency.
                continue

            avail = int(max(0, min(safe_available[ally.id], ally.ships - spent.get(ally.id, 0) - strategic_reserve(ally))))
            if avail < MIN_LAUNCH:
                continue

            tt = travel_time(ally.x, ally.y, ally.radius, mine.x, mine.y, mine.radius, max(1, avail))
            if tt >= fall_turn - 0.5:
                continue
            if sun_blocked(ally.x, ally.y, mine.x, mine.y):
                continue

            send = min(avail, need)
            if send < MIN_LAUNCH:
                continue

            ang = safe_angle(ally.x, ally.y, mine.x, mine.y)
            tt_int = max(1, int(math.ceil(travel_time(ally.x, ally.y, ally.radius, mine.x, mine.y, mine.radius, send))))
            moves.append([ally.id, float(ang), int(send)])
            spent[ally.id] = spent.get(ally.id, 0) + int(send)
            planned_commits[mine.id].append((tt_int, player, int(send)))
            need -= send
            if need <= 0:
                break

    # ---------- compute attack budgets ----------
    total_my_ships = sum(p.ships for p in my_planets)
    total_attack_cap = total_my_ships * (
        GLOBAL_ATTACK_RATIO_EARLY if phase == "early"
        else GLOBAL_ATTACK_RATIO_MID if phase in ("mid", "defend")
        else GLOBAL_ATTACK_RATIO_LATE
    )

    # Source availability after reserve and after defense spends.
    available = {}
    for p in my_planets:
        cap = int(min(safe_available[p.id], max(0, p.ships - base_reserve[p.id])))
        cap -= spent.get(p.id, 0)
        available[p.id] = max(0, cap)

    total_committed_attack = 0

    # ---------- candidate evaluation ----------
    candidates_by_src = defaultdict(list)
    targets_pool = neutral_planets + enemy_planets

    # Phase-aware target preference.
    for src in sorted(my_planets, key=lambda p: p.ships - spent.get(p.id, 0), reverse=True):
        avail = available[src.id]
        if avail < MIN_LAUNCH:
            continue

        # Limit number of launches per planet per turn.
        max_launches = 2 if phase in ("early", "defend") else (3 if phase == "mid" else 4)
        reserve_buffer = 0 if phase == "late" else 2

        scored = []
        for tgt in targets_pool:
            if tgt.id == src.id:
                continue

            # Quick distance gate.
            d0 = dist(src.x, src.y, tgt.x, tgt.y)
            max_range = 34 + gp * 38
            if phase == "late":
                max_range = 78
            if tgt.id in comet_ids:
                max_range += 10
            if d0 > max_range:
                continue

            orbiting = is_orbiting(tgt)

            # Quick rough ETA using a mid-sized fleet estimate.
            rough_ships = max(12, min(30, avail))
            ix, iy, tt = solve_intercept(src, tgt, omega, rough_ships, orbiting)
            if tt >= ATTACK_HORIZON:
                continue

            # Path checks.
            if sun_blocked(src.x, src.y, ix, iy):
                # Avoid obviously impossible lines.
                continue
            if planet_blocked(src.x, src.y, ix, iy, planets, src.id, tgt.id):
                continue

            eta = max(1, int(math.ceil(tt)))
            base_need = ships_needed_at_eta(
                tgt,
                eta,
                player,
                base_arrivals=ledger.get(tgt.id, []),
                planned_extra=planned_commits[tgt.id],
            )
            if base_need <= 0:
                continue

            # Target value: production + time value + ownership value.
            turns_left = max(1, remaining_steps - eta)
            value = tgt.production * float(turns_left)
            value += indirect_wealth(tgt, planets, player) * 0.18 * turns_left

            if tgt.owner == -1:
                value *= 1.25
            else:
                value *= 1.9
                if phase == "late":
                    value *= 1.18

            if tgt.production >= 4:
                value *= 1.12
            if orbiting:
                value *= 1.07
            if tgt.id in comet_ids:
                value *= 1.08

            # Early game: prioritize close, safe neutrals.
            if phase == "early" and tgt.owner == -1:
                if d0 > 26:
                    value *= 0.76
                if tgt.production <= 1:
                    value *= 0.84

            # Midgame: prevent empty, overextended growth.
            if phase in ("mid", "defend"):
                if tgt.owner == -1:
                    value *= 0.90
                else:
                    value *= 1.08

            # Late game: prefer enemy elimination and high production.
            if phase == "late" and tgt.owner != player:
                value *= 1.20

            cost = max(1.0, base_need + eta * 1.9)
            score = value / cost

            # Extra bonuses / penalties.
            if tgt.owner == -1:
                my_t, en_t = relative_reaction_time(tgt, my_planets, enemy_planets)
                if my_t <= en_t - 2:
                    score *= 1.15
                elif abs(my_t - en_t) <= 2:
                    score *= 0.82
            else:
                score *= 1.15
                if tgt.production >= 3:
                    score *= 1.10

            # A few special heuristics.
            if phase == "early" and src.ships > 55 and tgt.owner == -1 and tgt.production >= 2:
                score *= 1.10
            if phase == "defend" and tgt.owner == -1:
                score *= 0.88
            if phase == "late" and tgt.owner != -1:
                score *= 1.12

            scored.append((score, tgt, ix, iy, tt, eta, base_need))

        scored.sort(key=lambda x: -x[0])
        candidates_by_src[src.id] = scored

    # ---------- attack execution with commitment tracking ----------
    src_launch_count = defaultdict(int)

    # Global budget depends on phase and how much defense we already spent.
    # This is the main fix for midgame over-spawning.
    for src in sorted(my_planets, key=lambda p: available[p.id], reverse=True):
        if available[src.id] < MIN_LAUNCH:
            continue
        if total_committed_attack >= total_attack_cap:
            break
        if src_launch_count[src.id] >= MAX_LAUNCHES_PER_SRC:
            continue

        scored = candidates_by_src.get(src.id, [])
        if not scored:
            continue

        src_rem = available[src.id]
        launches_left = MAX_LAUNCHES_PER_SRC - src_launch_count[src.id]

        for score, tgt, ix, iy, tt, eta, base_need in scored:
            if launches_left <= 0:
                break
            if src_rem < MIN_LAUNCH:
                break
            if total_committed_attack >= total_attack_cap:
                break

            # Re-check because previous launches may have changed commitments.
            latest_need = ships_needed_at_eta(
                tgt,
                eta,
                player,
                base_arrivals=ledger.get(tgt.id, []),
                planned_extra=planned_commits[tgt.id],
            )
            if latest_need <= 0:
                continue

            # Do not overcommit to the same target from the same source.
            send = max(MIN_LAUNCH, latest_need)
            send = min(send, src_rem)

            # Midgame safety: do not send too much from one planet unless winning hard.
            if phase in ("mid", "defend"):
                send = min(send, int(max(12, src_rem * 0.62)))
            elif phase == "early":
                send = min(send, int(max(10, src_rem * 0.55)))
            elif phase == "late":
                send = min(send, int(max(16, src_rem * 0.78)))

            if send < MIN_LAUNCH:
                continue

            # Ensure the planet itself stays alive.
            if src_rem - send < max(0, base_reserve[src.id] - reserve_buffer):
                continue

            ang = safe_angle(src.x, src.y, ix, iy)
            if sun_blocked(src.x, src.y, src.x + math.cos(ang) * 3.0, src.y + math.sin(ang) * 3.0):
                continue

            moves.append([src.id, float(ang), int(send)])
            src_rem -= int(send)
            available[src.id] = src_rem
            total_committed_attack += int(send)
            src_launch_count[src.id] += 1
            launches_left -= 1
            planned_commits[tgt.id].append((eta, player, int(send)))

            # If we just sent a large chunk, stop this source early to prevent overspread.
            if phase in ("early", "mid", "defend") and src_rem < base_reserve[src.id] + 10:
                break

    # ---------- simple redistribution / support in mid-late game ----------
    # move only clear surplus from rear planets toward the front.
    if phase in ("mid", "late") and len(my_planets) > 1 and enemy_planets:
        ref_set = enemy_planets + neutral_planets
        front_dist = {p.id: min(dist(p.x, p.y, t.x, t.y) for t in ref_set) for p in my_planets}
        front = min(my_planets, key=lambda p: front_dist[p.id])

        rear_candidates = sorted(my_planets, key=lambda p: -front_dist[p.id])
        for src in rear_candidates:
            if src.id == front.id:
                continue
            if available[src.id] < 18:
                continue

            # Only send from clearly rear planets.
            if front_dist[src.id] < front_dist[front.id] * (1.22 if phase == "late" else 1.30):
                continue

            # Choose a forward allied planet closer to the front.
            forward_allies = [p for p in my_planets if p.id != src.id and front_dist[p.id] < front_dist[src.id] * 0.82]
            if not forward_allies:
                forward_allies = [front]

            forward_allies.sort(key=lambda p: dist(src.x, src.y, p.x, p.y))
            tgt = forward_allies[0]
            if tgt.id == src.id:
                continue

            send_ratio = 0.45 if phase == "mid" else 0.55
            send = int(available[src.id] * send_ratio)
            if send < 8:
                continue

            ix, iy, tt = solve_intercept(src, tgt, omega, send, False)
            if tt > 40:
                continue
            if sun_blocked(src.x, src.y, ix, iy):
                continue

            ang = safe_angle(src.x, src.y, ix, iy)
            moves.append([src.id, float(ang), int(send)])
            available[src.id] -= int(send)

    # ---------- deduplicate same source/angle launches ----------
    dedup = {}
    for sid, ang, sh in moves:
        key = (sid, round(float(ang), 4))
        if key in dedup:
            dedup[key] = [sid, ang, dedup[key][2] + int(sh)]
        else:
            dedup[key] = [sid, ang, int(sh)]

    final_moves = []
    used_per_src = defaultdict(int)
    for sid, ang, sh in dedup.values():
        src = next((p for p in my_planets if p.id == sid), None)
        if src is None:
            continue
        max_allowed = int(src.ships) - used_per_src[sid]
        send = min(int(sh), max(0, max_allowed))
        if send >= MIN_LAUNCH:
            final_moves.append([sid, float(ang), int(send)])
            used_per_src[sid] += int(send)

    return final_moves

### Submit to competition

Now that we have a main.py, all you need to do is click "Submit to competition" on the right and watch your entry show up on the competition leaderboard! Best of luck!